In [ ]:
#Install necessary packages

# Install ultralytics (YOLOv8)
!pip install ultralytics --quiet

# If using Roboflow to host your dataset
!pip install roboflow --quiet

In [ ]:
from ultralytics import YOLO
from google.colab import drive
import os
from roboflow import Roboflow
from google.colab import userdata

In [ ]:
drive.mount('/content/drive')

In [ ]:
os.makedirs('/content/drive/MyDrive/yolov8_detectorv2/', exist_ok=True)

In [ ]:
!cd /content/drive/MyDrive/yolov8_detectorv2
data=userdata.get('ROBOFLOW_KEY')
os.environ["ROBOFLOW_KEY"] = data
api_key = os.environ.get("ROBOFLOW_KEY")

In [ ]:
os.makedirs('/content/drive/MyDrive/yolov8_detectorv2/models/', exist_ok=True)

# Download dataset to LOCAL storage (not Drive)
rf = Roboflow(api_key=api_key)
project = rf.workspace("4421objectdetector").project("objectdetectionmodel-8dbp1")
dataset = project.version(4).download("yolov8", location="/content/datasets")

model = YOLO('yolov8m.pt')

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name='custom_detector_compost_recycling',
    device=0,
    cache=True
)
run_dir = results.save_dir  # YOLO gives you the exact folder
best_model = f"{run_dir}/weights/best.pt"

!cp {best_model} /content/drive/MyDrive/yolov8_detectorv2/models/best.pt
print("✅ Training complete! Model saved to Drive.")


In [ ]:

# Test model with camera
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np

def take_photo():
    js = Javascript('''
    async function takePhoto() {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', 0.8);
    }
    ''')
    display(js)
    data = eval_js('takePhoto()')
    binary = b64decode(data.split(',')[1])
    return cv2.imdecode(np.frombuffer(binary, np.uint8), cv2.IMREAD_COLOR)

# Capture and detect
trained_model = YOLO('/content/drive/MyDrive/yolov8_detector/models/best.pt')
print("📸 Click 'Capture' button to take photo...")
img = take_photo()
cv2.imwrite('captured.jpg', img)
results = trained_model.predict('captured.jpg', save=True)
display(Image('runs/detect/predict/captured.jpg'))

In [ ]:
!cp /content/runs/detect/custom_detector_compost_recycling3/weights/best.pt \
    /content/drive/MyDrive/yolov8_detectorv2/models/best2.pt
